# AutoResearch Progress — UK Flood Insurance Cat Model

Replicates Karpathy's autoresearch analysis notebook.
Reads `results.tsv` and plots the experiment frontier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

df = pd.read_csv('results.tsv', sep='\t')
df['val_score'] = pd.to_numeric(df['val_score'], errors='coerce')
df['status'] = df['status'].str.strip().str.upper()

print(f'Total experiments: {len(df)}')
counts = df['status'].value_counts()
print(counts.to_string())
n_keep = counts.get('KEEP', 0)
n_decided = n_keep + counts.get('DISCARD', 0)
if n_decided > 0:
    print(f'Keep rate: {n_keep}/{n_decided} = {n_keep/n_decided:.1%}')
df.head(10)

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments ({len(kept)} total):\n')
for i, row in kept.iterrows():
    print(f'  #{i:3d}  score={row["val_score"]:.6f}  {row["description"]}')

## Val Score Over Time (lower is better)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
baseline_score = valid.loc[0, 'val_score']
best = valid[valid['status'] == 'KEEP']['val_score'].min()

# Only plot experiments near or below baseline
below = valid[valid['val_score'] <= baseline_score + 0.05]

disc = below[below['status'] == 'DISCARD']
ax.scatter(disc.index, disc['val_score'], c='#cccccc', s=12, alpha=0.5, zorder=2, label='Discarded')

kept_v = below[below['status'] == 'KEEP']
ax.scatter(kept_v.index, kept_v['val_score'], c='#2ecc71', s=50, zorder=4,
           label='Kept', edgecolors='black', linewidths=0.5)

kept_mask = valid['status'] == 'KEEP'
kept_idx = valid.index[kept_mask]
kept_scores = valid.loc[kept_mask, 'val_score']
running_min = kept_scores.cummin()
ax.step(kept_idx, running_min, where='post', color='#27ae60',
        linewidth=2, alpha=0.7, zorder=3, label='Running best')

for idx, score in zip(kept_idx, kept_scores):
    desc = str(valid.loc[idx, 'description']).strip()
    if len(desc) > 45:
        desc = desc[:42] + '...'
    ax.annotate(desc, (idx, score), textcoords='offset points',
                xytext=(6, 6), fontsize=8, color='#1a7a3a',
                alpha=0.9, rotation=30, ha='left', va='bottom')

ax.set_xlabel('Experiment #', fontsize=12)
ax.set_ylabel('val_score (lower is better)', fontsize=12)
ax.set_title(f'AutoResearch Progress: {len(df)} Experiments, {len(kept_v)} Kept Improvements', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.2)
margin = (baseline_score - best) * 0.15 if baseline_score != best else 0.1
ax.set_ylim(best - margin, baseline_score + margin)

plt.tight_layout()
plt.savefig('progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to progress.png')

## Summary Statistics

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
baseline = df.iloc[0]['val_score']
best_score = kept['val_score'].min()
best_row = kept.loc[kept['val_score'].idxmin()]

print(f'Baseline val_score: {baseline:.6f}')
print(f'Best val_score:     {best_score:.6f}')
print(f'Total improvement:  {baseline - best_score:.6f} ({(baseline - best_score)/baseline*100:.2f}%)')
print(f'Best experiment:    {best_row["description"]}')

print('\nTop improvements (each vs previous kept):')
kept['prev_score'] = kept['val_score'].shift(1)
kept['delta'] = kept['prev_score'] - kept['val_score']
hits = kept.iloc[1:].sort_values('delta', ascending=False)
print(f'{"Rank":>4}  {"Delta":>9}  {"Score":>10}  Description')
print('-'*70)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f'{rank:4d}  {row["delta"]:+.6f}  {row["val_score"]:.6f}  {row["description"]}')